In [1]:
import pandas as pd

In [2]:
results      = pd.read_csv("data/raw/results.csv")
fifa         = pd.read_csv("data/raw/fifa_ranking-2024-06-20.csv")
former_names = pd.read_csv("data/raw/former_names.csv")

In [3]:
results['date'] = pd.to_datetime(results['date'])
fifa['rank_date'] = pd.to_datetime(fifa['rank_date'])
print(f"قبل الفلتر: {len(results):,} مباراة")

قبل الفلتر: 49,071 مباراة


In [4]:
results = results[results['date'] >= '2010-01-01']

print(f"بعد الفلتر: {len(results):,} مباراة")

بعد الفلتر: 15,488 مباراة


In [5]:
name_map = dict(zip(former_names['former'], former_names['current']))

results['home_team'] = results['home_team'].replace(name_map)
results['away_team'] = results['away_team'].replace(name_map)

In [6]:
name_fixes = {
    'United States'                   : 'USA',
    'South Korea'                     : 'Korea Republic',
    'Iran'                            : 'IR Iran',
    'Czech Republic'                  : 'Czechia',
    'North Korea'                     : 'Korea DPR',
    'Cape Verde'                      : 'Cabo Verde',
    'Kyrgyzstan'                      : 'Kyrgyz Republic',
    'Gambia'                          : 'The Gambia',
    'Saint Kitts and Nevis'           : 'St. Kitts and Nevis',
    'Saint Vincent and the Grenadines': 'St. Vincent and the Grenadines',
    'Saint Lucia'                     : 'St. Lucia',
    'Taiwan'                          : 'Chinese Taipei',
    'Brunei'                          : 'Brunei Darussalam',
    'São Tomé and Príncipe'           : 'São Tomé e Príncipe',
}

In [7]:
print(f" القيم الناقصة في results:")
print(results.isnull().sum())

print(f" القيم الناقصة في fifa_ranking:")
print(fifa.isnull().sum())

 القيم الناقصة في results:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64
 القيم الناقصة في fifa_ranking:
rank               9
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64


In [8]:
results = results.dropna(subset=['home_score', 'away_score'])

print(f"بعد حذف الناقص: {len(results):,} مباراة")

بعد حذف الناقص: 15,488 مباراة


In [9]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 1    # فوز المضيف
    elif row['home_score'] == row['away_score']:
        return 0    # تعادل
    else:
        return -1   # فوز الضيف

results['result'] = results.apply(get_result, axis=1)

In [10]:
print(f" توزيع النتائج:")
print(f"   فوز المضيف  : {(results['result'] == 1).sum():,}")
print(f"   تعادل       : {(results['result'] == 0).sum():,}")
print(f"   فوز الضيف   : {(results['result'] == -1).sum():,}")

 توزيع النتائج:
   فوز المضيف  : 7,400
   تعادل       : 3,593
   فوز الضيف   : 4,495


In [11]:
# وزن البطولة 

In [12]:
def get_tournament_weight(tournament):
    if tournament == 'FIFA World Cup':
        return 4        # الأهم
    elif any(cup in tournament for cup in ['UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup', 'Gold Cup']):
        return 3        # بطولات قارية كبيرة
    elif 'qualification' in tournament.lower():
        return 2        # تصفيات
    elif tournament == 'Friendly':
        return 1        # ودية
    else:
        return 2        # باقي البطولات الرسمية

results['tournament_weight'] = results['tournament'].apply(get_tournament_weight)

print(results['tournament_weight'].value_counts().sort_index())

tournament_weight
1    4871
2    6876
3    3485
4     256
Name: count, dtype: int64


In [13]:

#  النموذج لا يفهم أسماء الفرق مثل "Brazil" أو "Germany"
#  هو يفهم الأرقام فقط
#  لذلك نحتاج نحول كل فريق إلى رقم يعبّر عن قوته

#  نستخدم ترتيب FIFA كرقم يعبّر عن قوة الفريق
#  مثال: البرازيل ترتيب 5 vs اليمن ترتيب 95
#  النموذج يفهم الفرق فوراً
#  ليش ما نستخدم الترتيب الحالي؟
#  لأن الترتيب يتغير مع الوقت — مثال:
#  الأرجنتين 2018 = ترتيب 5
#  الأرجنتين 2022 = ترتيب 3
#  الأرجنتين 2024 = ترتيب 1
#  لو استخدمنا الترتيب الحالي على مباراة 2018
#  النموذج يتعلم معلومة غلط!
#  لذلك نجيب الترتيب اللي كان وقت المباراة بالضبط
#
#  النتيجة النهائية:
#  كل مباراة رح يكون عندها عمودين جديدين:
#  home_rank = ترتيب الفريق المضيف وقت المباراة
#  away_rank = ترتيب الفريق الضيف وقت المباراة
#
#  مثال:
#  date        home_team  home_rank  away_team  away_rank
#  2014-06-20  Brazil     10         Germany    2
#  2018-07-15  France     7          Croatia    20
#  2022-11-22  Argentina  3          Mexico     13
# ============================================================

In [14]:
WC2026_TEAMS = [
    # المضيفون - HOSTS (3)
    'United States', 'Mexico', 'Canada',

    # UEFA - أوروبا (16)
    'France', 'Spain', 'England', 'Portugal',
    'Netherlands', 'Belgium', 'Germany', 'Croatia',
    'Switzerland', 'Austria', 'Norway', 'Scotland',
    'Bosnia and Herzegovina', 'Sweden', 'Turkey', 'Czechia',

    # CONMEBOL - أمريكا الجنوبية (6)
    'Argentina', 'Brazil', 'Uruguay', 'Colombia',
    'Ecuador', 'Paraguay',

    # AFC - آسيا (9)
    'Japan', 'IR Iran', 'Korea Republic', 'Australia',
    'Saudi Arabia', 'Qatar', 'Uzbekistan', 'Jordan', 'Iraq',

    # CAF - أفريقيا (10)
    'Morocco', 'Senegal', 'Egypt', 'Algeria',
    "Côte d'Ivoire", 'Ghana', 'Cabo Verde',
    'Tunisia', 'South Africa', 'DR Congo',

    # CONCACAF - غير المضيفين (3)
    'Haiti', 'Panama', 'Curacao',

    # OFC - أوقيانوسيا (1)
    'New Zealand',
]

print(len(WC2026_TEAMS))  # 48 ✅
# فلتر الداتا على هذه المنتخبات فقط
before = len(results)
results = results[
    results['home_team'].isin(WC2026_TEAMS) |
    results['away_team'].isin(WC2026_TEAMS)
]
print(f"بعد فلتر منتخبات كأس العالم:")
print(f"   قبل : {before:,} مباراة")
print(f"   بعد : {len(results):,} مباراة")


48
بعد فلتر منتخبات كأس العالم:
   قبل : 15,488 مباراة
   بعد : 6,554 مباراة


In [15]:
fifa = fifa.sort_values('rank_date')

In [16]:
def get_fifa_rank(team, match_date):
    team_ranks = fifa[
        (fifa['country_full'] == team) &
        (fifa['rank_date'] <= match_date)
    ]
    if len(team_ranks) == 0:
        return None  
    return team_ranks.iloc[-1]['rank']

In [17]:
results['home_rank'] = results.apply(
    lambda row: get_fifa_rank(row['home_team'], row['date']), axis=1
)

results['away_rank'] = results.apply(
    lambda row: get_fifa_rank(row['away_team'], row['date']), axis=1
)
results = results.dropna(subset=['home_rank', 'away_rank'])

print(f"✅ تم الربط — {len(results):,} مباراة")

✅ تم الربط — 5,679 مباراة


In [18]:
print(results[['date','home_team','home_rank','away_team','away_rank','result']].head(10).to_string(index=False))

      date home_team  home_rank away_team  away_rank  result
2010-01-02     Qatar       86.0      Mali       47.0       0
2010-01-04     Egypt       24.0      Mali       47.0       1
2010-01-05     Ghana       34.0    Malawi       99.0       0
2010-01-06    Kuwait      104.0 Australia       21.0       0
2010-01-06  Thailand      105.0    Jordan      111.0       0
2010-01-06     Yemen      130.0     Japan       43.0      -1
2010-01-11    Malawi       99.0   Algeria       26.0       1
2010-01-12     Egypt       24.0   Nigeria       22.0       1
2010-01-13    Zambia       84.0   Tunisia       53.0       0
2010-01-14      Mali       47.0   Algeria       26.0      -1


In [19]:
#سبب المشكلة انه اسماء المنتخبات تختلف في كل  ملف
#في فرقة غير رسمية وما رح تشارك بكاس العالم لذلك رح احذف الداتا تاعتها

In [20]:
# احذف المباريات اللي فيها 100 افتراضي
before = len(results)
results = results[
    (results['home_rank'] != 100) &
    (results['away_rank'] != 100)
]
after = len(results)

print(f"قبل الحذف : {before:,}")
print(f"بعد الحذف : {after:,}")
print(f"المحذوف   : {before - after:,} مباراة")

# احفظ
results.to_csv("data/clean/results_clean.csv", index=False)
print(f"\n✅ تم الحفظ — البيانات نظيفة 100%!")

قبل الحذف : 5,679
بعد الحذف : 5,635
المحذوف   : 44 مباراة

✅ تم الحفظ — البيانات نظيفة 100%!


In [21]:
shootouts = pd.read_csv('data/raw/shootouts.csv')
shootouts['date'] = pd.to_datetime(shootouts['date'])  

results = results.merge(
    shootouts[['date', 'home_team', 'away_team', 'winner']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

results.rename(columns={'winner': 'shootout_winner'}, inplace=True)

print(results['shootout_winner'].notna().sum())  
"""
في الأدوار الإقصائية، لما تنتهي المباراة بالتعادل مثلاً 1-1، الموديل بدون هذا العمود ما بيعرف مين فاز فعلياً. الان بعد الدمج، الموديل يعرف إن الفائز كان فريق معين عن طريق ركلات الترجيح.
هذا مهم لما تجي تحسب الـ features زي win rate وform لكل فريق — بدونه رح تحسبها غلط على المباريات اللي انتهت بالركلات."""

109


'\nفي الأدوار الإقصائية، لما تنتهي المباراة بالتعادل مثلاً 1-1، الموديل بدون هذا العمود ما بيعرف مين فاز فعلياً. الان بعد الدمج، الموديل يعرف إن الفائز كان فريق معين عن طريق ركلات الترجيح.\nهذا مهم لما تجي تحسب الـ features زي win rate وform لكل فريق — بدونه رح تحسبها غلط على المباريات اللي انتهت بالركلات.'

In [22]:
# ============================================================
#  تقرير التحقق النهائي الشامل
# ============================================================

print("=" * 55)
print("📋 تقرير التحقق الكامل")
print("=" * 55)

# ── 1. عدد المباريات ─────────────────────────────────────────
print(f"\n1️⃣  عدد المباريات : {len(results):,}")

# ── 2. التواريخ ──────────────────────────────────────────────
print(f"\n2️⃣  أقدم مباراة  : {results['date'].min().date()}")
print(f"    أحدث مباراة  : {results['date'].max().date()}")

# ── 3. الأعمدة ───────────────────────────────────────────────
print(f"\n3️⃣  الأعمدة:")
for col in results.columns:
    print(f"    ✅ {col}")

# ── 4. القيم الناقصة ─────────────────────────────────────────
print(f"\n4️⃣  القيم الناقصة:")
nulls = results.isnull().sum()
if nulls.sum() == 0:
    print("    ✅ لا يوجد قيم ناقصة")
else:
    print(nulls[nulls > 0])

# ── 5. توزيع النتائج ─────────────────────────────────────────
print(f"\n5️⃣  توزيع النتائج:")
print(f"    فوز المضيف : {(results['result'] ==  1).sum():,}")
print(f"    تعادل      : {(results['result'] ==  0).sum():,}")
print(f"    فوز الضيف  : {(results['result'] == -1).sum():,}")

# ── 6. أوزان البطولات ────────────────────────────────────────
print(f"\n6️⃣  أوزان البطولات:")
print(f"    وزن 3 (كأس العالم) : {(results['tournament_weight'] == 3).sum():,}")
print(f"    وزن 2 (رسمية)      : {(results['tournament_weight'] == 2).sum():,}")
print(f"    وزن 1 (ودية)       : {(results['tournament_weight'] == 1).sum():,}")

# ── 7. ترتيب FIFA ────────────────────────────────────────────
print(f"\n7️⃣  ترتيب FIFA:")
print(f"    أعلى ترتيب     : {results['home_rank'].min():.0f}")
print(f"    أدنى ترتيب     : {results['home_rank'].max():.0f}")
print(f"    100 افتراضي    : {((results['home_rank']==100)|(results['away_rank']==100)).sum()}")

# ── 8. أكثر المنتخبات مباريات ────────────────────────────────
print(f"\n8️⃣  أكثر 10 منتخبات مباريات:")
all_teams = pd.concat([
    results['home_team'],
    results['away_team']
]).value_counts().head(10)
print(all_teams.to_string())

# ── 9. عينة من البيانات ──────────────────────────────────────
print(f"\n9️⃣  عينة من البيانات:")
print(results[['date','home_team','home_rank',
               'away_team','away_rank',
               'tournament_weight','result']].head(5).to_string(index=False))

print("\n" + "=" * 55)
print("✅ كل شيء تمام — جاهز للخطوة القادمة!")
print("=" * 55)

📋 تقرير التحقق الكامل

1️⃣  عدد المباريات : 5,635

2️⃣  أقدم مباراة  : 2010-01-02
    أحدث مباراة  : 2026-01-26

3️⃣  الأعمدة:
    ✅ date
    ✅ home_team
    ✅ away_team
    ✅ home_score
    ✅ away_score
    ✅ tournament
    ✅ city
    ✅ country
    ✅ neutral
    ✅ result
    ✅ tournament_weight
    ✅ home_rank
    ✅ away_rank
    ✅ shootout_winner

4️⃣  القيم الناقصة:
shootout_winner    5526
dtype: int64

5️⃣  توزيع النتائج:
    فوز المضيف : 2,685
    تعادل      : 1,318
    فوز الضيف  : 1,632

6️⃣  أوزان البطولات:
    وزن 3 (كأس العالم) : 1,389
    وزن 2 (رسمية)      : 2,158
    وزن 1 (ودية)       : 1,884

7️⃣  ترتيب FIFA:
    أعلى ترتيب     : 1
    أدنى ترتيب     : 211
    100 افتراضي    : 0

8️⃣  أكثر 10 منتخبات مباريات:
Mexico          245
Qatar           214
Jordan          210
Saudi Arabia    206
Argentina       206
France          205
Iraq            199
Spain           198
Germany         198
Brazil          194

9️⃣  عينة من البيانات:
      date home_team  home_rank away_team 

In [23]:
#feature engineering

In [24]:
def get_form(team, date, df, n=5):
    home = df[(df['home_team'] == team) & (df['date'] < date)].tail(n)
    away = df[(df['away_team'] == team) & (df['date'] < date)].tail(n)
    
    wins = (home['result'] == 1).sum() + (away['result'] == -1).sum()
    total = len(home) + len(away)
    
    return wins / total if total > 0 else 0.5

results['home_form'] = results.apply(lambda r: get_form(r['home_team'], r['date'], results), axis=1)
results['away_form'] = results.apply(lambda r: get_form(r['away_team'], r['date'], results), axis=1)

print(results[['home_team', 'away_team', 'home_form', 'away_form']].tail(5))

       home_team away_team  home_form  away_form
5630     Morocco   Senegal        0.7        0.9
5631     Bolivia    Panama        0.3        0.6
5632      Panama    Mexico        0.5        0.3
5633     Bolivia    Mexico        0.3        0.4
5634  Uzbekistan  China PR        0.4        0.0


In [25]:
print((results['home_form'] == 0.5).sum())
print((results['away_form'] == 0.5).sum())

903
851


In [26]:
def get_h2h(home_team, away_team, date, df, n=10):
    past = df[
        (df['date'] < date) & (
            ((df['home_team'] == home_team) & (df['away_team'] == away_team)) |
            ((df['home_team'] == away_team) & (df['away_team'] == home_team))
        )
    ].tail(n)
    
    if len(past) == 0:
        return 0.5
    
    home_wins = 0
    for _, row in past.iterrows():
        if row['home_team'] == home_team and row['result'] == 1:
            home_wins += 1
        elif row['home_team'] == away_team and row['result'] == -1:
            home_wins += 1
    
    return home_wins / len(past)

results['h2h'] = results.apply(
    lambda r: get_h2h(r['home_team'], r['away_team'], r['date'], results), axis=1
)

print(results[['home_team', 'away_team', 'h2h']].tail(5))
# الناتج هون بعطي نسبة فوز  الفريق المضيف 

       home_team away_team       h2h
5630     Morocco   Senegal  0.666667
5631     Bolivia    Panama  0.000000
5632      Panama    Mexico  0.000000
5633     Bolivia    Mexico  0.000000
5634  Uzbekistan  China PR  0.428571


In [27]:
def get_avg_goals(team, date, df, n=5):
    home = df[(df['home_team'] == team) & (df['date'] < date)].tail(n)
    away = df[(df['away_team'] == team) & (df['date'] < date)].tail(n)
    
    scored = list(home['home_score']) + list(away['away_score'])
    conceded = list(home['away_score']) + list(away['home_score'])
    
    avg_scored = sum(scored) / len(scored) if scored else 1.0
    avg_conceded = sum(conceded) / len(conceded) if conceded else 1.0
    
    return avg_scored, avg_conceded

results['home_avg_scored'], results['home_avg_conceded'] = zip(
    *results.apply(lambda r: get_avg_goals(r['home_team'], r['date'], results), axis=1)
)

results['away_avg_scored'], results['away_avg_conceded'] = zip(
    *results.apply(lambda r: get_avg_goals(r['away_team'], r['date'], results), axis=1)
)

print(results[['home_team', 'away_team', 'home_avg_scored', 'home_avg_conceded', 'away_avg_scored', 'away_avg_conceded']].tail(5))
#  جمعت الاهداف اخر خمس مباريات وقسمتهم على   عدد المباريات 

       home_team away_team  home_avg_scored  home_avg_conceded  \
5630     Morocco   Senegal              1.8                0.4   
5631     Bolivia    Panama              0.6                2.1   
5632      Panama    Mexico              1.6                0.7   
5633     Bolivia    Mexico              0.6                1.9   
5634  Uzbekistan  China PR              1.4                0.7   

      away_avg_scored  away_avg_conceded  
5630              3.0                0.3  
5631              1.7                0.6  
5632              0.8                1.1  
5633              0.9                1.0  
5634              0.6                2.4  


In [28]:
# فرق الترتيب بين الفريقين
# سالب = المضيف أقوى، موجب = الضيف أقوى
results['rank_diff'] = results['home_rank'] - results['away_rank']

print(results[['home_team', 'away_team', 'home_rank', 'away_rank', 'rank_diff']].tail(5))

       home_team away_team  home_rank  away_rank  rank_diff
5630     Morocco   Senegal       12.0       18.0       -6.0
5631     Bolivia    Panama       84.0       43.0       41.0
5632      Panama    Mexico       43.0       15.0       28.0
5633     Bolivia    Mexico       84.0       15.0       69.0
5634  Uzbekistan  China PR       62.0       88.0      -26.0


In [29]:
results.to_csv('data/clean/results_features.csv', index=False)
print("تم الحفظ ✅")
print(f"الأعمدة: {list(results.columns)}")

تم الحفظ ✅
الأعمدة: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'result', 'tournament_weight', 'home_rank', 'away_rank', 'shootout_winner', 'home_form', 'away_form', 'h2h', 'home_avg_scored', 'home_avg_conceded', 'away_avg_scored', 'away_avg_conceded', 'rank_diff']


In [30]:
wc_scorers = pd.read_csv(r"C:\Users\user\Desktop\project\data\raw\Players.Goal.csv", encoding='utf-8-sig')


In [31]:
print(wc_scorers.columns.tolist())
# المفروض يطلع: ['Player', 'Position', 'Goals']

['Player', 'Position', 'Goals']


In [32]:
position_fixes = {
    'Cristiano Ronaldo': 'Forward',
    'Romelu Lukaku': 'Forward',
    'Robert Lewandowski': 'Forward',
}

for player, correct_pos in position_fixes.items():
    wc_scorers.loc[wc_scorers['Player'] == player, 'Position'] = correct_pos

print("تم تصليح المراكز ✅")
print(wc_scorers[wc_scorers['Player'].isin(position_fixes.keys())].to_string(index=False))

تم تصليح المراكز ✅
            Player Position  Goals
 Cristiano Ronaldo  Forward    128
     Romelu Lukaku  Forward     83
Robert Lewandowski  Forward     82


In [33]:
wc_scorers.to_csv(r"C:\Users\user\Desktop\project\data\raw\Players.Goal.csv", index=False, encoding='utf-8-sig')
print("تم حفظ الملف ✅")

تم حفظ الملف ✅


In [34]:
goals = pd.read_csv(r"C:\Users\user\Desktop\project\data\raw\goalscorers.csv")

In [35]:
# نشوف اللاعبين اللي عندهم مشكلة - هل موجودين في goalscorers بنفس الاسم؟
problematic = wc_scorers[wc_scorers['Player'].str.contains('\?', na=False)]['Player'].tolist()

for player in problematic:
    match = goals[goals['scorer'].str.contains(player.split('?')[0], na=False)]['scorer'].unique()
    if len(match) > 0:
        print(f"Players.Goal: {player}")
        print(f"goalscorers : {match}")
        print()

Players.Goal: Andrej Kramari?
goalscorers : ['Andrej Kramarić']



<>:2: SyntaxWarning: invalid escape sequence '\?'
<>:2: SyntaxWarning: invalid escape sequence '\?'
C:\Users\user\AppData\Local\Temp\ipykernel_25256\519680735.py:2: SyntaxWarning: invalid escape sequence '\?'
  problematic = wc_scorers[wc_scorers['Player'].str.contains('\?', na=False)]['Player'].tolist()


Players.Goal: Luka Modri?
goalscorers : ['Luka Modrić']

Players.Goal: Marcelo Brozovi?
goalscorers : ['Marcelo Brozović']

Players.Goal: Mateo Kova?i?
goalscorers : ['Mateo Kovačić']

Players.Goal: Haris Seferovi?
goalscorers : ['Haris Seferovic']

Players.Goal: Piotr Zieli?ski
goalscorers : ['Piotr Zieliński']

Players.Goal: Karol ?widerski
goalscorers : ['Karol Jokl' 'Karol Dobiaš' 'Karol Kisel' 'Karol Linetty'
 'Karol Świderski' 'Karol Mets']

Players.Goal: Jakub Kami?ski
goalscorers : ['Jakub Kamiński']

Players.Goal: Krzysztof Pi?tek
goalscorers : ['Krzysztof Piątek']

Players.Goal: Aleksandar Mitrovi?
goalscorers : ['Aleksandar Mitrović']

Players.Goal: Sergej Milinkovi?-Savi?
goalscorers : ['Sergej Milinković-Savić']

Players.Goal: Luka Jovi?
goalscorers : ['Luka Jović']

Players.Goal: Filip Kosti?
goalscorers : ['Filip Kostić']

Players.Goal: Bruno Petkovi?
goalscorers : ['Bruno Petković']

Players.Goal: Jakub Kami?ski
goalscorers : ['Jakub Kamiński']

Players.Goal: Sebastian 

In [36]:
name_fixes = {
    'Andrej Kramari?': 'Andrej Kramarić',
    'Luka Modri?': 'Luka Modrić',
    'Marcelo Brozovi?': 'Marcelo Brozović',
    'Mateo Kova?i?': 'Mateo Kovačić',
    'Haris Seferovi?': 'Haris Seferovic',  # موحد مع goalscorers
    'Piotr Zieli?ski': 'Piotr Zieliński',
    'Karol ?widerski': 'Karol Świderski',
    'Jakub Kami?ski': 'Jakub Kamiński',
    'Krzysztof Pi?tek': 'Krzysztof Piątek',
    'Aleksandar Mitrovi?': 'Aleksandar Mitrović',
    'Sergej Milinkovi?-Savi?': 'Sergej Milinković-Savić',
    'Luka Jovi?': 'Luka Jović',
    'Filip Kosti?': 'Filip Kostić',
    'Bruno Petkovi?': 'Bruno Petković',
    'Sebastian Szyma?ski': 'Sebastian Szymański',
    'Przemys?aw Frankowski': 'Przemysław Frankowski',
    'Strahinja Pavlovi?': 'Strahinja Pavlović',
    'Nemanja Radonji?': 'Nemanja Radonjić',
}

wc_scorers['Player'] = wc_scorers['Player'].replace(name_fixes)
print("تم تصليح الأسماء ✅")
print(wc_scorers[wc_scorers['Player'].str.contains('\?', na=False)])

تم تصليح الأسماء ✅
               Player    Position  Goals
83       Ivan Peri�i?     Forward     33
88      Mario Pa�ali?     Forward     10
150    Du�an Vlahovi?  Midfielder     13
153       Du�an Tadi?  Midfielder     23
293     Nikola Vla�i?     Forward      7
389  Andrija �ivkovi?  Midfielder      1


<>:24: SyntaxWarning: invalid escape sequence '\?'
<>:24: SyntaxWarning: invalid escape sequence '\?'
C:\Users\user\AppData\Local\Temp\ipykernel_25256\691002321.py:24: SyntaxWarning: invalid escape sequence '\?'
  print(wc_scorers[wc_scorers['Player'].str.contains('\?', na=False)])


In [37]:
remaining = ['Ivan Peri', 'Mario Pa', 'Du', 'Nikola Vla', 'Andrija']

for name in remaining:
    match = goals[goals['scorer'].str.contains(name, na=False)]['scorer'].unique()
    if len(match) > 0:
        print(f"goalscorers: {match}")

goalscorers: ['Ivan Perišić']
goalscorers: ['Mario Pašalić']
goalscorers: ['José Durand Laguna' 'Ernesto Duarte Machado da Silva' 'Paddy Duncan'
 'Gösta Dunker' 'Jimmy Dunne' 'Harry Duggan' 'Duberty Aráoz'
 'Jorge Duilio Benitez' 'Duncan Edwards' 'Peter Ducke' 'Roland Ducke'
 'Lascelles Dunkley' 'Edy Dublin' 'Dušan Kabát' 'Emil Dumitriu'
 'Alan Durban' 'Florea Dumitrache' 'Antal Dunai' 'Gilbert Dussier'
 'Tony Dunne' 'Ibrahim Al Duraihem' 'Dušan Bajević' 'Michael Dupuy'
 'Ion Dumitru' 'Dumitru Marcu' 'Dudu Georgescu' 'Dušan Galis'
 'Kama Dumbuya' 'Dušan Savić' 'Duh Deng-Sheng' 'Duncan Cole'
 'Rodolfo Dubó' 'Robbie Dunn' 'Malcolm Dunford' 'Gordon Durie'
 'Jean-Philippe Durand' 'Duddley Hatei' 'Francis Dupont'
 'Mehmet Durakovic' 'Peter Dubovský' 'Ilie Dumitrescu' 'Itumeleng Duiker'
 'Asam Duraiban' 'Carlos Dunga' 'Duncan Shearer' 'Dušan Tittel'
 'Christophe Dugarry' 'Rafael Dudamel' 'Dusit Chalermsan' 'Le Huynh Duc'
 'Liuai Duguga' 'Dunga' 'Dumisa Ngobe' 'Duilio Davino' 'Richard Dunne'


In [38]:
name_fixes2 = {
    'Ivan Peri\x8di?': 'Ivan Perišić',
    'Mario Pa\x8dali?': 'Mario Pašalić',
    'Du\x8dan Vlahovi?': 'Dušan Vlahović',
    'Du\x8dan Tadi?': 'Dušan Tadić',
    'Nikola Vla\x8di?': 'Nikola Vlašić',
    'Andrija \x8divkovi?': 'Andrija Živković',
}

wc_scorers['Player'] = wc_scorers['Player'].replace(name_fixes2)

# تحقق إذا باقي مشاكل
remaining = wc_scorers[wc_scorers['Player'].str.contains(r'\?', regex=True, na=False)]
print(f"لاعبين باقيين فيهم مشكلة: {len(remaining)}")
print(remaining)

لاعبين باقيين فيهم مشكلة: 6
               Player    Position  Goals
83       Ivan Peri�i?     Forward     33
88      Mario Pa�ali?     Forward     10
150    Du�an Vlahovi?  Midfielder     13
153       Du�an Tadi?  Midfielder     23
293     Nikola Vla�i?     Forward      7
389  Andrija �ivkovi?  Midfielder      1


In [39]:
wc_scorers.loc[83, 'Player'] = 'Ivan Perišić'
wc_scorers.loc[88, 'Player'] = 'Mario Pašalić'
wc_scorers.loc[150, 'Player'] = 'Dušan Vlahović'
wc_scorers.loc[153, 'Player'] = 'Dušan Tadić'
wc_scorers.loc[293, 'Player'] = 'Nikola Vlašić'
wc_scorers.loc[389, 'Player'] = 'Andrija Živković'

remaining = wc_scorers[wc_scorers['Player'].str.contains(r'\?', regex=True, na=False)]
print(f"لاعبين باقيين فيهم مشكلة: {len(remaining)}")


لاعبين باقيين فيهم مشكلة: 0


In [40]:
wc_scorers.to_csv(r"C:\Users\user\Desktop\project\data\clean\Players.Goal.csv", index=False, encoding='utf-8-sig')
print("تم حفظ الملف في مجلد clean ✅")

تم حفظ الملف في مجلد clean ✅
